# Steel Surface Defect Detector — Colab Training

This notebook trains the steel-defect classifier end-to-end on a GPU runtime:
data prep → 5-fold cross-validation training → Grad-CAM → ONNX export → live Gradio demo.

**Before running:** `Runtime -> Change runtime type -> GPU` (T4 is fine).

Two ways to get the code onto Colab:
1. Push this project to your own GitHub repo and `git clone` it below (recommended), or
2. Upload the project zip via the Colab file browser and unzip it (cell provided below, commented out).


## 1. Get the code

In [ ]:
# Option A: clone your own repo (edit the URL)
# !git clone https://github.com/<your-username>/steel-defect-detector.git
# %cd steel-defect-detector

# Option B: if you uploaded steel-defect-detector.zip via the Colab Files pane, uncomment:
# !unzip -q steel-defect-detector.zip -d .
# %cd steel-defect-detector

import os
print("Working dir:", os.getcwd())


## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt


## 3. Get the NEU-CLS / NEU-DET data

Pick ONE of the options below.

**Option A — Kaggle (recommended, matches the layout `dataset.py` expects most closely).**
Requires a free Kaggle account. Upload your `kaggle.json` API token when prompted.


In [ ]:
from google.colab import files
import os

if not os.path.exists("/root/.kaggle/kaggle.json"):
    print("Upload your kaggle.json (Kaggle account -> Settings -> API -> Create New Token)")
    uploaded = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    for fname in uploaded:
        os.rename(fname, "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets download -d kaustubhdikshit/neu-surface-defect-database -p raw_neu --unzip
!python scripts/prepare_data.py --input raw_neu --output ./NEU-CLS --mode copy


**Option B — GitHub mirror (no Kaggle account needed).**
Skip this cell if Option A worked.


In [ ]:
# !git clone --depth 1 https://github.com/siddhartamukherjee/NEU-DET-Steel-Surface-Defect-Detection.git neu_mirror
# !python scripts/prepare_data.py --input neu_mirror --output ./NEU-CLS --val-frac 0.15 --mode copy


In [ ]:
# Sanity check: should show 1800 images, ~300 per class
from pathlib import Path
from src.dataset import load_all_paths
import numpy as np

paths, labels = load_all_paths(Path("NEU-CLS"))
print("Total images:", len(paths))
print("Per-class counts:", dict(zip(*np.unique(labels, return_counts=True))))


## 4. Train

Full run uses `configs/default.yaml` as-is: EfficientNet-B4, 30 epochs, 5-fold CV.
On a Colab T4 this is roughly 3-6 hours depending on queue/throttling. Start with the
quick run first to confirm everything works, then launch the full run.


In [ ]:
# Quick smoke test (~10-15 min on a T4) — confirms the whole pipeline works end-to-end
!python train.py --config configs/default.yaml --backbone efficientnet_b0 --epochs 5 --folds 2 --batch-size 32 --out-dir outputs_smoketest


In [ ]:
# Full run as specified in configs/default.yaml
!python train.py --config configs/default.yaml --out-dir outputs


In [ ]:
# If you want a faster full run: fewer folds / epochs, still gives a real cross-validated estimate
# !python train.py --config configs/default.yaml --folds 3 --epochs 20 --out-dir outputs


## 5. Inspect results

In [ ]:
import json
with open("outputs/cv_summary.json") as f:
    summary = json.load(f)
print(json.dumps(summary, indent=2))


In [ ]:
from IPython.display import Image, display
display(Image("outputs/fold0/confusion_matrix.png"))
display(Image("outputs/fold0/training_curves.png"))
display(Image("outputs/fold0/gradcam_samples.png"))


## 6. Export to ONNX

In [ ]:
!python -m src.export_onnx --checkpoint outputs/fold0/best_model.pt --out steel_defect.onnx


## 7. Launch the Gradio demo

`share=True` gives a public `gradio.live` link that works from Colab.


In [ ]:
!python app.py --checkpoint outputs/fold0/best_model.pt --share


## 8. Download your trained artifacts

Zips the checkpoints + plots + ONNX model so you can pull them off Colab.


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("steel_defect_results", "zip", "outputs")
files.download("steel_defect_results.zip")
files.download("steel_defect.onnx")
